# Check: does early stopping on log loss cripple the unweighted LightGBM?

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
df = pd.concat([pd.read_parquet(f"{PATH}/train.parquet"), pd.read_parquet(f"{PATH}/test.parquet")])
feats = [c for c in df.columns if c != "label"]
tr, rest = train_test_split(df.index, train_size=0.5, stratify=df["label"], random_state=0)
va, te = train_test_split(rest, train_size=0.4, stratify=df.loc[rest, "label"], random_state=0)
X, y = {}, {}
for k, v in [("train", tr), ("val", va), ("test", te)]:
    X[k], y[k] = df.loc[v, feats], df.loc[v, "label"]

In [ ]:
for weighted in [False, True]:
    ev = {}
    m = lgb.LGBMClassifier(objective="multiclass", class_weight="balanced" if weighted else None, learning_rate=0.08,
                           num_leaves=31, n_estimators=400, random_state=42, verbose=-1)
    m.fit(X["train"], y["train"], eval_set=[(X["val"], y["val"])], callbacks=[lgb.record_evaluation(ev)])
    loss = ev["valid_0"]["multi_logloss"]
    print(f"weighted={weighted} | val log loss at 1/2/5/10/50/100/400 trees:", [round(loss[i - 1], 4) for i in (1, 2, 5, 10, 50, 100, 400)],
          "| minimum at", int(np.argmin(loss)) + 1)
    for n in (1, 5, 20, 50, 100, 200, 400):
        pv, pt = m.predict(X["val"], num_iteration=n), m.predict(X["test"], num_iteration=n)
        print(f"  {n:>3} trees | val macro {f1_score(y['val'], pv, average='macro', zero_division=0):.4f} weighted {f1_score(y['val'], pv, average='weighted', zero_division=0):.4f}"
              f" | test macro {f1_score(y['test'], pt, average='macro', zero_division=0):.4f} weighted {f1_score(y['test'], pt, average='weighted', zero_division=0):.4f}", flush=True)